In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# ---- Load the data ----
# Replace the paths below if the CSVs are in a different folder
prices_df = pd.read_csv('prices-split-adjusted.csv')
fundamentals_df = pd.read_csv('fundamentals.csv')

# ---- Merge relevant columns ----
data_df = pd.merge(
    prices_df,
    fundamentals_df[['Ticker Symbol']],
    how='inner',
    left_on='symbol',
    right_on='Ticker Symbol'
)

# ---- Select features and target ----
X = data_df[['open', 'low', 'high']].values
y = data_df['close'].values

# ---- Train-test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Standardize features ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Build the neural network ----
model = Sequential()
model.add(Dense(32, input_dim=X_train_scaled.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))  # Output layer for regression

# ---- Compile the model ----
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

# ---- Train the model ----
model.fit(
    X_train_scaled, y_train,
    epochs=5,
    batch_size=32,
    verbose=1
)

# ---- Evaluate on test data ----
loss = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nMean Squared Error on Test Set: {loss:.4f}")

# ---- Predictions ----
y_pred = model.predict(X_test_scaled)

# Print a few example predictions
print("\nExample Predictions:")
for i in range(5):
    print(f"Predicted: {y_pred[i][0]:.4f}  |  Actual: {y_test[i]:.4f}")

# Overall MSE
mse = np.mean((y_pred - y_test) ** 2)
print(f"\nCalculated Mean Squared Error: {mse:.4f}")

Epoch 1/5


C:\Users\MANI BUTWALL\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


75849/75849 ━━━━━━━━━━━━━━━━━━━━ 189s 2ms/step - loss: 256.3752
Epoch 2/5
75849/75849 ━━━━━━━━━━━━━━━━━━━━ 192s 3ms/step - loss: 0.8836
Epoch 3/5
75849/75849 ━━━━━━━━━━━━━━━━━━━━ 183s 2ms/step - loss: 0.7365
Epoch 4/5
75849/75849 ━━━━━━━━━━━━━━━━━━━━ 225s 3ms/step - loss: 0.6846
Epoch 5/5
75849/75849 ━━━━━━━━━━━━━━━━━━━━ 186s 2ms/step - loss: 0.5901
